# CET-Epi: Intervention Simulation

Test scale-aware intervention propagation.

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.cet_epi import CET_Epi
from src.data.chickenpox_loader import MultiScaleChickenpoxLoader
from src.evaluation.intervention import InterventionSimulator
from src.utils.config import load_config
from src.utils.gpu import setup_gpu

# Setup
device = setup_gpu()
config = load_config('chickenpox.yaml', '../configs')

# Load model
checkpoint = torch.load('../checkpoints/chickenpox/best_model.pt', map_location=device)
model = CET_Epi(
    n_micro=config.data.micro_nodes,
    n_macro=config.data.macro_nodes,
    in_channels=config.data.features,
    hidden_dim=config.model.hidden_dim
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

simulator = InterventionSimulator(model, device)

## Compare Intervention Strategies

In [ ]:
# Load test data
loader = MultiScaleChickenpoxLoader(lags=config.data.get('lags', 4))
_, test_data = loader.get_split(config.data.train_ratio)
sample = next(iter(test_data))

x = sample.x.to(device)
edge_index = sample.edge_index.to(device)
edge_attr = sample.edge_attr.to(device) if sample.edge_attr is not None else None

# Define strategies
strategies = {
    'lockdown_high_center': {'nodes': [0, 1, 2, 3, 4], 'effect': 0.8},
    'lockdown_medium_center': {'nodes': [0, 1, 2, 3, 4], 'effect': 0.5},
    'lockdown_low_center': {'nodes': [0, 1, 2, 3, 4], 'effect': 0.2},
    'lockdown_high_periphery': {'nodes': [15, 16, 17, 18, 19], 'effect': 0.8},
    'vaccination_cluster': {'nodes': [8, 9, 10, 11], 'effect': 0.6},
}

results = simulator.compare_intervention_strategies(x, edge_index, strategies, edge_attr)

# Visualization
names = list(results.keys())
micro_red = [results[n]['micro_reduction']*100 for n in names]
macro_red = [results[n]['macro_reduction']*100 for n in names]
ratios = [results[n]['propagation_ratio'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x_pos = np.arange(len(names))
width = 0.35

axes[0].bar(x_pos - width/2, micro_red, width, label='Micro Reduction', alpha=0.8)
axes[0].bar(x_pos + width/2, macro_red, width, label='Macro Reduction', alpha=0.8)
axes[0].set_ylabel('Reduction (%)')
axes[0].set_title('Intervention Effects by Scale')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(x_pos, ratios, color='green', alpha=0.7)
axes[1].axhline(y=1.0, color='red', linestyle='--', label='Perfect Propagation')
axes[1].set_ylabel('Propagation Ratio')
axes[1].set_title('Cross-Scale Propagation Efficiency')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(names, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../logs/figures/intervention_comparison.png', dpi=300, bbox_inches='tight')
plt.show()